In [4]:
def build_A_Q_for_p(p_true: float):
    theta_prep = 2 * math.asin(math.sqrt(p_true))
    theta_amp = theta_prep / 2

    A = QuantumCircuit(1, name="A")
    A.ry(theta_prep, 0)

    Q = QuantumCircuit(1, name="Q")
    Q.ry(theta_prep, 0)
    Q.z(0)
    Q.ry(-theta_prep, 0)
    Q.x(0); Q.z(0); Q.x(0)

    return A, Q, theta_amp

def estimate_p_for_k(p_true: float, k: int, shots: int):
    A, Q, theta_amp = build_A_Q_for_p(p_true)

    qc = QuantumCircuit(1, 1)
    qc.compose(A, inplace=True)
    for _ in range(k):
        qc.compose(Q, inplace=True)
    qc.measure(0, 0)

    result = sampler.run([qc], shots=shots).result()
    quasi = result.quasi_dists[0]
    p_k_est = quasi.get(1, 0.0)

    eps = 1e-12
    p_k_clipped = min(max(p_k_est, eps), 1 - eps)

    alpha = math.asin(math.sqrt(p_k_clipped))

    theta1 = alpha / (2*k + 1)
    theta2 = (math.pi - alpha) / (2*k + 1)

    p1 = math.sin(theta1)**2
    p2 = math.sin(theta2)**2

    p_hat = p1 if abs(p1 - p_true) < abs(p2 - p_true) else p2

    return p_hat, p_k_est


In [5]:
p_true = 0.1
k = 1
shot_values = [1000, 2000, 5000, 10000, 20000, 50000]
R = 20  # number of repeated runs for variance estimation

shot_results = []

for shots in shot_values:
    errors = []
    for _ in range(R):
        p_hat, _ = estimate_p_for_k(p_true, k, shots)
        errors.append(abs(p_hat - p_true))

    shot_results.append({
        "shots": shots,
        "mean_error": np.mean(errors),
        "std_error": np.std(errors),
        "all_errors": errors
    })

shot_results


NameError: name 'QuantumCircuit' is not defined

In [6]:
shots = [r["shots"] for r in shot_results]
mean_err = [r["mean_error"] for r in shot_results]
std_err = [r["std_error"] for r in shot_results]

plt.figure(figsize=(7,5))
plt.plot(shots, mean_err, marker='o')
plt.xscale('log')
plt.yscale('log')
plt.xlabel("Number of shots (log scale)")
plt.ylabel("Mean absolute error (log scale)")
plt.title("QAE finite-shot error decay (p_true = 0.1, k = 1)")
plt.grid(True)
plt.show()

plt.figure(figsize=(7,5))
plt.plot(shots, std_err, marker='o')
plt.xscale('log')
plt.yscale('log')
plt.xlabel("Number of shots (log scale)")
plt.ylabel("Std dev of absolute error (log scale)")
plt.title("QAE variance vs shots")
plt.grid(True)
plt.show()


ValueError: Data has no positive values, and therefore cannot be log-scaled.

<Figure size 700x500 with 1 Axes>

ValueError: Data has no positive values, and therefore cannot be log-scaled.

<Figure size 700x500 with 1 Axes>

In [7]:
import sys
!{sys.executable} -m pip install numpy matplotlib --quiet

407.15s - pydevd: Sending message related to process being replaced timed-out after 5 seconds



[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 4.4 — Finite-Shot Convergence Validation
# Drop this into your Jupyter notebook, replacing the existing shot validation
# Uses your exact build_A_Q_for_p and estimate_p_for_k functions unchanged
# ─────────────────────────────────────────────────────────────────────────────

import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ── parameters ────────────────────────────────────────────────────────────────
p_true = 0.1
k      = 1
R      = 50   # repeats per shot count — increase from 20 for smoother curves

# Extended shot range: 8,192 now sits in the middle, not at the edge
# Covers 512 → 65,536 in approximately equal log-space steps
shot_values = [512, 1024, 2048, 4096, 8192, 16384, 32768, 65536]

# ── data collection (uses your existing estimate_p_for_k unchanged) ───────────
shot_results = []
for shots in shot_values:
    errors = []
    for _ in range(R):
        p_hat, _ = estimate_p_for_k(p_true, k, shots)
        errors.append(abs(p_hat - p_true))
    shot_results.append({
        "shots":      shots,
        "mean_error": np.mean(errors),
        "std_error":  np.std(errors),
        "all_errors": errors,
    })

# ── extract arrays ────────────────────────────────────────────────────────────
shots_arr = np.array([r["shots"]      for r in shot_results])
mean_err  = np.array([r["mean_error"] for r in shot_results])
std_err   = np.array([r["std_error"]  for r in shot_results])

# ── log-log slope fit (mean error) ────────────────────────────────────────────
slope, intercept = np.polyfit(np.log10(shots_arr), np.log10(mean_err), 1)
print(f"Fitted log-log slope (mean error): {slope:.4f}  (theory: -0.50)")

# ── % change at 8192 → 16384 ──────────────────────────────────────────────────
idx_8k  = list(shot_values).index(8192)
idx_16k = list(shot_values).index(16384)
pct = (mean_err[idx_16k] - mean_err[idx_8k]) / mean_err[idx_8k] * 100
print(f"Mean error at  8,192 shots : {mean_err[idx_8k]:.3e}")
print(f"Mean error at 16,384 shots : {mean_err[idx_16k]:.3e}")
print(f"Reduction 8k → 16k         : {pct:.1f}%  (theory: -29.3% = 1 - 1/sqrt(2))")

# ── theoretical O(N^{-1/2}) reference anchored at first point ─────────────────
ref_mean = mean_err[0] * np.sqrt(shots_arr[0] / shots_arr)
ref_std  = std_err[0]  * np.sqrt(shots_arr[0] / shots_arr)

# ── plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    r"Figure 5: Finite-shot convergence behaviour for fixed "
    r"$p_{\rm true} = 0.1$, $k = 1$",
    fontsize=13, fontweight="bold"
)

xtick_labels = [f"{s:,}" for s in shot_values]

for ax, y, y_ref, ylabel, title_letter, title_desc in [
    (axes[0], mean_err, ref_mean,
     "Mean absolute error (log scale)",
     "a", "Mean absolute error vs $N_{\\rm shot}$ (log–log)"),
    (axes[1], std_err, ref_std,
     "Std dev of absolute error (log scale)",
     "b", "Standard deviation vs $N_{\\rm shot}$ (log–log)"),
]:
    ax.loglog(shots_arr, y,     'o-',  color='steelblue', linewidth=1.8,
              markersize=6, label="Empirical")
    ax.loglog(shots_arr, y_ref, 'k--', linewidth=1.2, alpha=0.7,
              label=r"$\mathcal{O}(N_{\rm shot}^{-1/2})$ theory")

    # Mark 8,192
    ax.axvline(8192, color='red', linestyle=':', linewidth=1.5,
               label=r"$N_{\rm shots} = 8{,}192$ (Ch. 5 budget)")
    ax.annotate("8,192", xy=(8192, y.max()), xytext=(8192 * 1.08, y.max()),
                color='red', fontsize=8.5, va='top')

    ax.set_xlabel(r"Number of shots $N_{\rm shot}$ (log scale)", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(f"({title_letter}) {title_desc}", fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, which='both', alpha=0.3)
    ax.set_xticks(shots_arr)
    ax.set_xticklabels(xtick_labels, rotation=35, fontsize=8)

plt.tight_layout()
plt.savefig("finite_shot_validation.png", dpi=150, bbox_inches='tight')
plt.show()
print("Done.")

NameError: name 'QuantumCircuit' is not defined